In [1]:
import pandas as pd
import os
from openai import AsyncOpenAI

## Real Data

In [2]:
data = pd.read_csv("../data/processed/train.csv")

In [3]:
client = AsyncOpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [4]:
import asyncio
from tqdm import tqdm

model = "gpt-5.4-mini-2026-03-17"

semaphore = asyncio.Semaphore(20)

decoding_params = {
    "temperature": 0.2,
    "max_completion_tokens": 30
}

PROMPT_TEMPLATE = """
You are analyzing Russian VK-style social media comments.

Task:
Assign EXACTLY ONE category describing the TYPE of the comment.

Choose the MOST dominant category that best describes the main intent of the text.

Categories:

- short reaction
- emotional outburst
- complaint / frustration
- insult / toxic message
- personal story
- daily life update
- romantic / relationships
- humor / joke / meme
- philosophical / reflective
- poetry / creative writing
- opinion / discussion
- question / asking
- other

Rules:
- Choose the closest category
- Do NOT invent new categories
- Be consistent

Text:
{txt}

Return ONLY the category name.
"""

VALID_TOPICS = {
    "short reaction",
    "emotional outburst",
    "complaint / frustration",
    "insult / toxic message",
    "personal story",
    "daily life update",
    "romantic / relationships",
    "humor / joke / meme",
    "philosophical / reflective",
    "poetry / creative writing",
    "opinion / discussion",
    "question / asking",
    "other"
}

def normalize_topic(t):
    t = t.strip().lower()
    return t if t in VALID_TOPICS else "other"


async def get_topic(text):
    prompt = PROMPT_TEMPLATE.format(txt=text)

    try:
        async with semaphore:
            response = await client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                **decoding_params
            )

        raw = response.choices[0].message.content
        return normalize_topic(raw)

    except Exception:
        return "other"


async def process_all(texts, batch_size=50):
    results = []

    for i in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[i:i+batch_size]
        tasks = [get_topic(t) for t in batch_texts]

        batch_results = await asyncio.gather(*tasks)
        results.extend(batch_results)

    return results


In [8]:
# ===== SAMPLE (50% stratified) =====
sample_df = data.groupby("label", group_keys=False).apply(
    lambda x: x.sample(frac=0.5, random_state=42)
).reset_index(drop=True)

texts = sample_df["text"].astype(str).tolist()

# ===== RUN =====
topics = await process_all(texts)

sample_df["topic"] = topics

/var/folders/ct/6y14r9zn1kj3j9kqvchn0lsm0000gn/T/ipykernel_47672/3479359459.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sample_df = data.groupby("label", group_keys=False).apply(
  0%|          | 0/47 [00:00<?, ?it/s]

100%|██████████| 47/47 [02:53<00:00,  3.70s/it]


In [9]:
# ===== ANALYSIS =====
print("\nTOPIC DISTRIBUTION:")
print(sample_df["topic"].value_counts(normalize=True).head(20))

print("\nTOPIC vs LABEL:")
print(pd.crosstab(sample_df["topic"], sample_df["label"], normalize="index"))


TOPIC DISTRIBUTION:
topic
other                         0.177281
short reaction                0.125645
humor / joke / meme           0.114888
philosophical / reflective    0.092943
opinion / discussion          0.091652
insult / toxic message        0.080895
complaint / frustration       0.063683
emotional outburst            0.061102
daily life update             0.058950
romantic / relationships      0.040017
question / asking             0.036575
poetry / creative writing     0.028399
personal story                0.027969
Name: proportion, dtype: float64

TOPIC vs LABEL:
label                       negative   neutral  positive
topic                                                   
complaint / frustration     0.783784  0.202703  0.013514
daily life update           0.036496  0.649635  0.313869
emotional outburst          0.612676  0.232394  0.154930
humor / joke / meme         0.071161  0.726592  0.202247
insult / toxic message      0.813830  0.170213  0.015957
opinion / discuss

### COMMENT TYPE DISTRIBUTION — Findings

- The most common category is **"other"** (~17.7%), indicating that a portion of texts does not fit predefined categories  

- The dataset is dominated by short and informal communication:
  - Short reactions (~12.6%)  
  - Humor / memes (~11.5%)  
  - Opinion / discussion (~9.2%)  
  - Philosophical / reflective (~9.3%)  

- Toxic and negative communication is also substantial:
  - Insult / toxic (~8.1%)  
  - Complaint / frustration (~6.4%)  
  - Emotional outbursts (~6.1%)  

- Less frequent but important categories:
  - Daily life updates (~5.9%)  
  - Romantic (~4.0%)  
  - Questions (~3.7%)  
  - Creative / poetry (~2.8%)  
  - Personal stories (~2.8%)  

### Interpretation

- The dataset reflects a **highly heterogeneous and informal communication space**, typical for social media  

- A large portion of texts are:
  - short  
  - reactive  
  - emotionally expressive  

- At the same time, there is a mix of:
  - structured content (opinions, stories)  
  - creative content (poetry)  

- The relatively high share of "other" suggests:
  - either limitations of the category set  
  - or the presence of hybrid / ambiguous comment types  

---

### COMMENT TYPE vs SENTIMENT — Findings

- Negative sentiment is strongly associated with:
  - Insult / toxic messages (~81%)  
  - Complaint / frustration (~78%)  
  - Emotional outbursts (~61%)  

- Positive sentiment is dominated by:
  - Romantic / relationships (~73%)  
  - Short reactions (~45%)  
  - Daily life updates (~31%)  

- Neutral sentiment is associated with:
  - Questions (~93%)  
  - Humor / memes (~73%)  
  - Opinion / discussion (~67%)  
  - Philosophical texts (~62%)  

- Mixed categories:
  - Personal stories → distributed across all classes  
  - Poetry → mostly neutral but with noticeable negative share  

### Interpretation

- Sentiment is strongly dependent on **comment type**, not only content  

- Negative sentiment is driven by:
  - aggression  
  - frustration  
  - emotional expression  

- Neutral sentiment often corresponds to:
  - informational or observational content  
  - humor and discussion  

- Positive sentiment is linked to:
  - interpersonal interaction  
  - short expressive reactions  

- This suggests that:
  - sentiment classification is partially a **functional classification problem**  
  - different discourse types exhibit different sentiment distributions  

- Therefore, ignoring comment structure may limit model performance  

---

### Key Insight

- Real data is not only diverse in length and vocabulary, but also in **functional types of communication**  

- These types have distinct sentiment patterns  

- This further supports earlier findings that:
  - synthetic data must capture not only surface statistics  
  - but also underlying communication structure  

## Synthetic Data

In [10]:
data_synthetic = pd.read_csv("../data_synthetic/synthetic_decoding_params_1_5k.csv")
data_synthetic

,text,label
0,"Сделали по-своему, без лишнего пафоса — и здра...",neutral
1,"видел щас это в ленте, просто листнул дальше и...",neutral
2,"У нас вот целый день дождь с ветром, до магази...",neutral
3,"Опять этот трэш... как же з..ло уже, всё через...",negative
4,Да сколько можно уже этой фигнёй кормить... вс...,negative
...,...,...
1495,опять эти снег/то ли дождь… обувь уже сдает(-:,neutral
1496,"Ооой, кайфец тaкой 😍 прям залип безответно... ...",positive
1497,"ой, забавно вышло, если честно просто мимоходо...",neutral
1498,"Ооо дааа, кайф вообще 😍🔥",positive


In [11]:
# ===== SAMPLE (50% stratified) =====
sample_df_synthetic = data_synthetic.groupby("label", group_keys=False).apply(
    lambda x: x.sample(frac=0.5, random_state=42)
).reset_index(drop=True)

texts_synthetic = sample_df_synthetic["text"].astype(str).tolist()

# ===== RUN =====
topics_synthetic = await process_all(texts_synthetic)

sample_df_synthetic["topic"] = topics_synthetic

/var/folders/ct/6y14r9zn1kj3j9kqvchn0lsm0000gn/T/ipykernel_47672/4125919266.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sample_df_synthetic = data_synthetic.groupby("label", group_keys=False).apply(
100%|██████████| 15/15 [00:49<00:00,  3.33s/it]


In [ ]:
print("\nSYNTHETIC - TOPIC DISTRIBUTION:")
print(sample_df_synthetic["topic"].value_counts(normalize=True))

print("\nSYNTHETIC - TOPIC vs LABEL:")
print(pd.crosstab(sample_df_synthetic["topic"], sample_df_synthetic["label"], normalize="index"))


SYNTHETIC — TOPIC DISTRIBUTION:
topic
short reaction                0.384000
complaint / frustration       0.341333
opinion / discussion          0.100000
philosophical / reflective    0.056000
daily life update             0.052000
humor / joke / meme           0.037333
personal story                0.017333
emotional outburst            0.006667
insult / toxic message        0.002667
question / asking             0.001333
other                         0.001333
Name: proportion, dtype: float64

SYNTHETIC — TOPIC vs LABEL:
label                       negative   neutral  positive
topic                                                   
complaint / frustration     0.804688  0.195312  0.000000
daily life update           0.000000  1.000000  0.000000
emotional outburst          0.800000  0.000000  0.200000
humor / joke / meme         0.000000  0.964286  0.035714
insult / toxic message      1.000000  0.000000  0.000000
opinion / discussion        0.000000  1.000000  0.000000
other         

In [13]:
real_dist = sample_df["topic"].value_counts(normalize=True)
synthetic_dist = sample_df_synthetic["topic"].value_counts(normalize=True)

compare_df = pd.concat([real_dist, synthetic_dist], axis=1)
compare_df.columns = ["real", "synthetic"]
compare_df = compare_df.fillna(0)

compare_df.sort_values("real", ascending=False)

,real,synthetic
topic,,
other,0.177281,0.001333
short reaction,0.125645,0.384000
humor / joke / meme,0.114888,0.037333
philosophical / reflective,0.092943,0.056000
opinion / discussion,0.091652,0.100000
insult / toxic message,0.080895,0.002667
complaint / frustration,0.063683,0.341333
emotional outburst,0.061102,0.006667
daily life update,0.058950,0.052000


In [14]:
import plotly.express as px

compare_df_plot = compare_df.reset_index().rename(columns={"index": "topic"})

px.bar(
    compare_df_plot,
    x="topic",
    y=["real", "synthetic"],
    barmode="group",
    title="Topic Distribution: Real vs Synthetic"
).show()

### COMMENT TYPE DISTRIBUTION: REAL vs SYNTHETIC - Findings

- Synthetic data exhibits a drastically different distribution of comment types compared to real data  

- Strong overrepresentation in synthetic data:
  - Short reactions: 12.6% → 38.4%  
  - Complaint / frustration: 6.4% → 34.1%  

- Strong underrepresentation or near absence:
  - Insult / toxic: 8.1% → 0.3%  
  - Emotional outburst: 6.1% → 0.7%  
  - Romantic: 4.0% → 0.0%  
  - Poetry / creative: 2.8% → 0.0%  
  - Questions: 3.7% → 0.1%  

- Humor / memes are also significantly reduced:
  - 11.5% → 3.7%  

- The "other" category almost disappears:
  - 17.7% → 0.1%  

- Opinion / discussion remains relatively stable:
  - ~9–10%  

### Interpretation

- Synthetic data collapses the diversity of communication types into a few dominant patterns:
  - short reactions  
  - complaints  

- Complex and expressive forms of communication are almost entirely lost:
  - toxicity and insults  
  - emotional outbursts  
  - creative and poetic texts  
  - interpersonal (romantic) interactions  

- This indicates that LLM-based generation:
  - fails to reproduce the full spectrum of real user behavior  
  - strongly biases toward “safe” and structurally simple outputs  

- The near absence of toxic and emotionally extreme content suggests:
  - implicit alignment constraints of the model  
  - avoidance of aggressive or offensive language  

- The disappearance of "other" indicates:
  - lack of unpredictability and edge cases in synthetic data  

### Key Insight

- Real data contains a rich distribution of **communication types**, each with distinct linguistic and emotional characteristics  

- Synthetic data fails to capture this structure and instead produces a **compressed and biased representation of discourse**

- This suggests that:
  - synthetic data generation should account not only for surface features (length, vocabulary)  
  - but also for **functional types of communication**

- Ignoring this aspect may lead to:
  - loss of important training signals  
  - reduced model robustness on real-world data  

### Check "Other" categories group

In [15]:
other_df = sample_df[sample_df["topic"] == "other"]

print(len(other_df))
other_df["text"].sample(20, random_state=42).tolist()

412


['Я люблю тебя маленькая , прости меня ... Плохо без тебя',
 'в школе тебе бы поучиться',
 'Сочинские монетки и купюры) Мой папа молодец))',
 'Зря сняли Красножана. Локомотив без него в призеры не попадет, а договорной был матч с Анжи или нет еще не известно. Хотя, конечно, причиной отставки могут быть совершенно другие причины, а это лишь формальный повод.',
 'ахахах Мишенька, солнышко!!видос пятибальный))) \nтеперь я знаю ты потаенные желания...))))))',
 '...Вчера мне исполнилось 30 лет,я встретил и провел этот день на море)))...',
 'ахахах.. лол(',
 'Почему Русь хотят уничтожить...если об этом узнает мир:',
 'джорджо скоро лоло приходит,загудим',
 '"О чем бы написать,что я уже не тот,да вроде тот,просто время идет вперед..."',
 'Меня окружают 4 типа людей,которые постоянно поют:\n1.Барбара Стрейзанд..ууууууууууууу..\n2.Чумачечая веснаа....чумачечаая\n3.Все тип-топ каблучки макияяж\n4. И которые петь не любят',
 'Всем хорошего настроения!!!)))))))',
 'Очень по тебе скучаю, братишка :

In [16]:
sample_texts = other_df["text"].sample(50, random_state=42).tolist()

prompt = f"""
You are analyzing Russian social media comments.

Below are comments that were classified as "other".

Your task:
1. Identify common patterns in these comments
2. Suggest 3–7 NEW categories that better describe them

Comments:
{sample_texts}

Return:
- list of patterns
- list of proposed categories (short names)
"""

response = await client.chat.completions.create(
    model="gpt-5.4-mini-2026-03-17",
    messages=[{"role": "user", "content": prompt}],
    temperature=0.3,
    max_completion_tokens=300
)

print(response.choices[0].message.content)

### Common patterns in these “other” comments
- **Emotional reactions and personal feelings**: love, missing someone, gratitude, sadness, excitement, congratulations.
- **Informal slang / internet-style speech**: “ахахаха”, “лол”, “DDD”, emojis, stretched words, playful spelling.
- **Short reactions to media/content**: “круто”, “шикарная песня”, “веселое видео”, “фотка охуенчик”.
- **Personal life updates / status posts**: birthday, trip, school, army, baptism, graduation, wedding-related moments.
- **Social interaction / addressing someone directly**: comments to friends, family, or named people.
- **Humor, irony, and memes**: jokes, sarcastic remarks, playful one-liners.
- **Mixed-topic commentary**: sports, politics, music, local events, but usually in a casual, non-analytic way.
- **Spam / moderation / promotional notes**: “свои рекламы прошу не размещать”, links, invite-style content.

### Proposed new categories
1. **Эмоциональные реакции**
2. **Личные обновления**
3. **Оценка ко

### Analysis of "Other" Category

- The "other" category accounts for a significant portion of real data (~17.7%)  

- Manual inspection and LLM-assisted analysis reveal that these comments are not random noise, but rather a mixture of:

  - short emotional reactions  
  - informal slang and fragmented expressions  
  - personal updates and social interactions  
  - humor and casual commentary  
  - meta-level or context-dependent messages  

- Many of these texts:
  - combine multiple communication functions  
  - are too short or ambiguous for precise categorization  
  - rely on context or shared understanding  

### Interpretation

- The "other" category captures **edge cases and hybrid communication types** that do not fit cleanly into predefined categories  

- This reflects the inherently messy and informal nature of real-world social media data  

- In contrast, synthetic data almost completely lacks such cases:
  - indicating a lack of unpredictability and mixed communication patterns  
  - further supporting the conclusion that synthetic data is overly structured and simplified  

### Key Insight

- The presence of a large "other" category is not a limitation, but a **feature of real data complexity**  

- Synthetic data fails to reproduce this aspect, resulting in:
  - reduced diversity of communication patterns  
  - loss of edge cases and ambiguous examples  